# Visualization 1 – Evolution of baby names over time
### French baby names (1900–2020) — Altair 6

Implementation of Week 1 Design A: Heatmap of name popularity over time.

In [1]:
import altair as alt
import pandas as pd

alt.data_transformers.enable('json')

DataTransformerRegistry.enable('json')

In [2]:
names_raw = pd.read_csv('Names hints/dpt2020.csv', sep=';', encoding='utf-8')
names_raw.drop(names_raw[names_raw.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names_raw.drop(names_raw[names_raw.dpt == 'XX'].index, inplace=True)

names_raw['annais'] = pd.to_numeric(names_raw['annais'], errors='coerce')
names_raw = names_raw.dropna(subset=['annais'])
names_raw['annais'] = names_raw['annais'].astype(int)

national = names_raw.groupby(['preusuel', 'annais', 'sexe'])['nombre'].sum().reset_index()

print(f'Data: {len(names_raw):,} rows, {national["preusuel"].nunique():,} unique names')
print(f'Years: {names_raw.annais.min()} to {names_raw.annais.max()}')

Data: 3,668,274 rows, 15,270 unique names
Years: 1900 to 2020


---
## Chart 1 — Top 30 names all time (1900–2020)

**Each row** = a name sorted by its peak year | **Each column** = a year | **Color** = % of national births

Color scale is capped at 6% to improve contrast for names with 1–4% popularity (MARIE reaches 12.74% but is clamped to dark red).

---
## Chart 2 — Top 30 names since 1990

Shows the contemporary names absent from Chart 1: EMMA, LUCAS, LÉA, CAMILLE, HUGO...

Maximum popularity in this period is ~2% — confirming the explosion of naming diversity after 1990.

In [3]:
# ── total births per year (all time) ──────────────────────────────────────
total_per_year = (
    national.groupby('annais')['nombre'].sum()
    .reset_index()
    .rename(columns={'nombre': 'total_year'})
)

# ── CHART 1 : top 30 all-time ──────────────────────────────────────────────
top30 = national.groupby('preusuel')['nombre'].sum().nlargest(30).index.tolist()

heatmap_data = (
    national[national['preusuel'].isin(top30)]
    .groupby(['preusuel', 'annais'])['nombre'].sum()
    .reset_index()
    .merge(total_per_year, on='annais')
)
heatmap_data['pct'] = (heatmap_data['nombre'] / heatmap_data['total_year'] * 100).round(4)

peak_years = (
    heatmap_data.loc[heatmap_data.groupby('preusuel')['pct'].idxmax()]
    [['preusuel', 'annais']]
    .sort_values('annais')
)
name_order = peak_years['preusuel'].tolist()

viz1 = (
    alt.Chart(heatmap_data)
    .mark_rect()
    .encode(
        x=alt.X('annais:O', title='Year',
                axis=alt.Axis(
                    labelExpr="datum.value % 10 == 0 ? datum.value : ''",
                    labelAngle=-45
                )),
        y=alt.Y('preusuel:N', title='Name', sort=name_order),
        color=alt.Color('pct:Q',
                        scale=alt.Scale(scheme='reds', domainMax=6, clamp=True),
                        title='% of national births (capped at 6%)'),
        tooltip=[
            alt.Tooltip('preusuel:N', title='Name'),
            alt.Tooltip('annais:O',   title='Year'),
            alt.Tooltip('nombre:Q',   title='Births', format=',d'),
            alt.Tooltip('pct:Q',      title='% national', format='.3f'),
        ]
    )
    .properties(
        width=900, height=520,
        title=alt.TitleParams(
            'Chart 1 — Top 30 names all time (1900–2020) | color capped at 6% for better contrast',
            fontSize=13
        )
    )
)

# ── CHART 2 : top 30 since 1990 ───────────────────────────────────────────
national_recent = national[national['annais'] >= 1990]

total_per_year_r = (
    national_recent.groupby('annais')['nombre'].sum()
    .reset_index()
    .rename(columns={'nombre': 'total_year'})
)

top30_recent = national_recent.groupby('preusuel')['nombre'].sum().nlargest(30).index.tolist()

heatmap_recent = (
    national_recent[national_recent['preusuel'].isin(top30_recent)]
    .groupby(['preusuel', 'annais'])['nombre'].sum()
    .reset_index()
    .merge(total_per_year_r, on='annais')
)
heatmap_recent['pct'] = (heatmap_recent['nombre'] / heatmap_recent['total_year'] * 100).round(4)

peak_years_r = (
    heatmap_recent.loc[heatmap_recent.groupby('preusuel')['pct'].idxmax()]
    [['preusuel', 'annais']]
    .sort_values('annais')
)
name_order_r = peak_years_r['preusuel'].tolist()

viz1b = (
    alt.Chart(heatmap_recent)
    .mark_rect()
    .encode(
        x=alt.X('annais:O', title='Year', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('preusuel:N', title='Name', sort=name_order_r),
        color=alt.Color('pct:Q',
                        scale=alt.Scale(scheme='blues', zero=True),
                        title='% of national births (1990–2020)'),
        tooltip=[
            alt.Tooltip('preusuel:N', title='Name'),
            alt.Tooltip('annais:O',   title='Year'),
            alt.Tooltip('nombre:Q',   title='Births', format=',d'),
            alt.Tooltip('pct:Q',      title='% national', format='.3f'),
        ]
    )
    .properties(
        width=400, height=520,
        title=alt.TitleParams(
            'Chart 2 — Top 30 names since 1990 | max ~2%: naming diversity has massively increased',
            fontSize=13
        )
    )
)

alt.vconcat(viz1, viz1b).resolve_scale(color='independent')


alt.VConcatChart(...)